In [3]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.metrics import accuracy_score
import numpy as np

In [4]:
df=pd.read_csv('/content/crop_yield.csv')

In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000000 entries, 0 to 999999
Data columns (total 10 columns):
 #   Column                  Non-Null Count    Dtype  
---  ------                  --------------    -----  
 0   Region                  1000000 non-null  object 
 1   Soil_Type               1000000 non-null  object 
 2   Crop                    1000000 non-null  object 
 3   Rainfall_mm             1000000 non-null  float64
 4   Temperature_Celsius     1000000 non-null  float64
 5   Fertilizer_Used         1000000 non-null  bool   
 6   Irrigation_Used         1000000 non-null  bool   
 7   Weather_Condition       1000000 non-null  object 
 8   Days_to_Harvest         1000000 non-null  int64  
 9   Yield_tons_per_hectare  1000000 non-null  float64
dtypes: bool(2), float64(3), int64(1), object(4)
memory usage: 62.9+ MB


In [6]:
df.duplicated().sum()

np.int64(0)

In [7]:
df.describe()

,Rainfall_mm,Temperature_Celsius,Days_to_Harvest,Yield_tons_per_hectare
count,1000000.000000,1000000.000000,1000000.000000,1000000.000000
mean,549.981901,27.504965,104.495025,4.649472
std,259.851320,7.220608,25.953412,1.696572
min,100.000896,15.000034,60.000000,-1.147613
25%,324.891090,21.254502,82.000000,3.417637
50%,550.124061,27.507365,104.000000,4.651808
75%,774.738520,33.753267,127.000000,5.879200
max,999.998098,39.999997,149.000000,9.963372


In [8]:
df['Fertilizer_Used'] = df['Fertilizer_Used'].astype(int)
df['Irrigation_Used'] = df['Irrigation_Used'].astype(int)

In [9]:
df.head()

,Region,Soil_Type,Crop,Rainfall_mm,Temperature_Celsius,Fertilizer_Used,Irrigation_Used,Weather_Condition,Days_to_Harvest,Yield_tons_per_hectare
0,West,Sandy,Cotton,897.077239,27.676966,0,1,Cloudy,122,6.555816
1,South,Clay,Rice,992.673282,18.026142,1,1,Rainy,140,8.527341
2,North,Loam,Barley,147.998025,29.794042,0,0,Sunny,106,1.127443
3,North,Sandy,Soybean,986.866331,16.644190,0,1,Rainy,146,6.517573
4,South,Silt,Wheat,730.379174,31.620687,1,1,Cloudy,110,7.248251


In [10]:
df.drop('Region',axis=1,inplace=True)

In [11]:
df.drop('Weather_Condition',axis=1,inplace=True)

In [12]:
df_sampled = df.sample(n=1000000, random_state=42).reset_index(drop=True)

In [13]:
df.head()

,Soil_Type,Crop,Rainfall_mm,Temperature_Celsius,Fertilizer_Used,Irrigation_Used,Days_to_Harvest,Yield_tons_per_hectare
0,Sandy,Cotton,897.077239,27.676966,0,1,122,6.555816
1,Clay,Rice,992.673282,18.026142,1,1,140,8.527341
2,Loam,Barley,147.998025,29.794042,0,0,106,1.127443
3,Sandy,Soybean,986.866331,16.644190,0,1,146,6.517573
4,Silt,Wheat,730.379174,31.620687,1,1,110,7.248251


In [14]:
features_to_scale = ['Rainfall_mm', 'Temperature_Celsius', 'Days_to_Harvest']

scaler = StandardScaler()

df_scaled = df_sampled.copy()
df_scaled[features_to_scale] = scaler.fit_transform(df_sampled[features_to_scale])

In [15]:
df_scaled.head()

,Soil_Type,Crop,Rainfall_mm,Temperature_Celsius,Fertilizer_Used,Irrigation_Used,Days_to_Harvest,Yield_tons_per_hectare
0,Silt,Cotton,0.634488,-0.502602,0,0,0.597416,3.840988
1,Chalky,Cotton,1.195387,-0.614085,0,0,-1.020869,5.138173
2,Sandy,Barley,0.970171,-0.482624,1,1,1.368028,6.401523
3,Chalky,Cotton,-1.332936,-1.469372,0,1,-0.327318,2.658805
4,Silt,Rice,-0.151832,-1.260568,0,1,-1.521767,2.797703


In [16]:
df_scaled = pd.get_dummies(df_scaled, columns=['Soil_Type', 'Crop'])

In [17]:
df_scaled.head()

,Rainfall_mm,Temperature_Celsius,Fertilizer_Used,Irrigation_Used,Days_to_Harvest,Yield_tons_per_hectare,Soil_Type_Chalky,Soil_Type_Clay,Soil_Type_Loam,Soil_Type_Peaty,Soil_Type_Sandy,Soil_Type_Silt,Crop_Barley,Crop_Cotton,Crop_Maize,Crop_Rice,Crop_Soybean,Crop_Wheat
0,0.634488,-0.502602,0,0,0.597416,3.840988,False,False,False,False,False,True,False,True,False,False,False,False
1,1.195387,-0.614085,0,0,-1.020869,5.138173,True,False,False,False,False,False,False,True,False,False,False,False
2,0.970171,-0.482624,1,1,1.368028,6.401523,False,False,False,False,True,False,True,False,False,False,False,False
3,-1.332936,-1.469372,0,1,-0.327318,2.658805,True,False,False,False,False,False,False,True,False,False,False,False
4,-0.151832,-1.260568,0,1,-1.521767,2.797703,False,False,False,False,False,True,False,False,False,True,False,False


In [18]:
df_scaled = df_scaled.replace({True: 1, False: 0})
df_scaled.head()

/tmp/ipython-input-317/4026290963.py:1: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_scaled = df_scaled.replace({True: 1, False: 0})


,Rainfall_mm,Temperature_Celsius,Fertilizer_Used,Irrigation_Used,Days_to_Harvest,Yield_tons_per_hectare,Soil_Type_Chalky,Soil_Type_Clay,Soil_Type_Loam,Soil_Type_Peaty,Soil_Type_Sandy,Soil_Type_Silt,Crop_Barley,Crop_Cotton,Crop_Maize,Crop_Rice,Crop_Soybean,Crop_Wheat
0,0.634488,-0.502602,0,0,0.597416,3.840988,0,0,0,0,0,1,0,1,0,0,0,0
1,1.195387,-0.614085,0,0,-1.020869,5.138173,1,0,0,0,0,0,0,1,0,0,0,0
2,0.970171,-0.482624,1,1,1.368028,6.401523,0,0,0,0,1,0,1,0,0,0,0,0
3,-1.332936,-1.469372,0,1,-0.327318,2.658805,1,0,0,0,0,0,0,1,0,0,0,0
4,-0.151832,-1.260568,0,1,-1.521767,2.797703,0,0,0,0,0,1,0,0,0,1,0,0


In [19]:
X = df_scaled.drop('Yield_tons_per_hectare', axis=1)


y = df_scaled['Yield_tons_per_hectare']

In [20]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [21]:
# 1. Initialize the Model
# We'll set n_jobs=-1 to use all your computer's processors for faster training
rf_model = RandomForestRegressor(n_estimators=200, max_depth=20, random_state=42, n_jobs=-1)

In [22]:
# 2. Train the model
rf_model.fit(X_train, y_train)

RandomForestRegressor(max_depth=20, n_estimators=200, n_jobs=-1,
                      random_state=42)

In [23]:
# 3. Evaluate Training Accuracy (to check for learning)
train_preds = rf_model.predict(X_train)
train_r2 = r2_score(y_train, train_preds)

In [24]:
# 4. Evaluate Testing Accuracy (to check for real-world performance)
test_preds = rf_model.predict(X_test)
test_r2 = r2_score(y_test, test_preds)
mae = mean_absolute_error(y_test, test_preds)

In [25]:
print(f"Training R2 Score: {train_r2:.4f}")
print(f"Testing R2 Score: {test_r2:.4f}")
print(f"Average Error: {mae:.2f} tons per hectare")

Training R2 Score: 0.9570
Testing R2 Score: 0.9100
Average Error: 0.41 tons per hectare


In [26]:
# 1. Define a function to calculate percentage accuracy
def get_accuracy(actual, predicted):
    mape = np.mean(np.abs((actual - predicted) / actual)) * 100
    accuracy = 100 - mape
    return accuracy

# 2. Get predictions
train_preds = rf_model.predict(X_train)
test_preds = rf_model.predict(X_test)

# 3. Calculate Accuracy %
train_acc = get_accuracy(y_train, train_preds)
test_acc = get_accuracy(y_test, test_preds)

print(f"Training Accuracy: {train_acc:.2f}%")
print(f"Testing Accuracy:  {test_acc:.2f}%")

Training Accuracy: 90.57%
Testing Accuracy:  87.64%
